# 59 — Cluster Optimization

**Repository:** `confidence-scheduled-decoding`

Notebook 59 closes the second systems arc. The design question shifts from local adaptive serving to cluster-level optimization:

\[
\max_{\pi, a, r}\; G(\pi, a, r)
\quad
\text{subject to bounded latency, memory pressure, verification load, and failure spillover.}
\]

The figures are synthetic specification diagrams. They make the control surfaces, routing policies, and system invariants explicit enough to implement next.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import ListedColormap, BoundaryNorm

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 13,
    "axes.titlesize": 28,
    "axes.labelsize": 18,
    "legend.fontsize": 12,
})

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches="tight")
    print(path)

def rounded_box(ax, xy, w, h, text, fontsize=14, lw=1.8):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=1.8, style="->", rad=0.0):
    ax.annotate(
        "", xy=end, xytext=start,
        arrowprops=dict(arrowstyle=style, lw=lw, color="black",
                        connectionstyle=f"arc3,rad={rad}")
    )

def flow_figure(title, labels, subtitle, fname, figsize=(13, 5), fontsize=13):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, pad=20)
    n = len(labels)
    w = 0.13 if n <= 6 else 0.105
    h = 0.18
    xs = np.linspace(0.07, 0.93-w, n)
    y = 0.52
    for i, (x, label) in enumerate(zip(xs, labels)):
        rounded_box(ax, (x, y), w, h, label, fontsize=fontsize)
        if i < n-1:
            arrow(ax, (x+w, y+h/2), (xs[i+1], y+h/2))
    rounded_box(ax, (0.22, 0.17), 0.56, 0.12, subtitle, fontsize=12)
    savefig(fname)
    plt.show()

## 1. Cluster optimization flow

In [ ]:
flow_figure(
    "Cluster optimization specifies global adaptive serving",
    ["distributed\nschedulers", "cluster\ntelemetry", "global\nobjective", "capacity\noptimizer", "recovery\npolicy", "optimized\nserving"],
    "Notebook 59 closes the second systems arc: local adaptive serving becomes cluster-level optimization.",
    "59_cluster_optimization_flow.png",
    figsize=(14, 5),
    fontsize=13
)

## 2. Cluster objective

In [ ]:
x = np.linspace(0, 1, 220)
throughput = 1.25 * (1 - np.exp(-3.4*x))
latency_penalty = 0.20 + 1.05*x**2.4
failure_penalty = 0.14 + 0.75*np.maximum(x-0.62, 0)**1.7
objective = throughput - 0.55*latency_penalty - 0.65*failure_penalty
best_x = x[np.argmax(objective)]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, throughput, label="throughput benefit", lw=2)
ax.plot(x, latency_penalty, label="latency pressure", lw=2)
ax.plot(x, failure_penalty, label="failure spillover risk", lw=2)
ax.plot(x, objective, label="cluster objective", lw=2)
ax.axvline(best_x, ls="--", color="black", label=f"best ≈ {best_x:.2f}")
ax.set_title("Cluster objective balances gain, latency, and spillover")
ax.set_xlabel("global optimization intensity")
ax.set_ylabel("normalized value")
ax.grid(alpha=0.3)
ax.legend(loc="best")
savefig("59_cluster_objective.png")
plt.show()

## 3. Capacity balance surface

In [ ]:
reserve = np.linspace(0.05, 0.95, 110)
load = np.linspace(0.05, 1.0, 110)
R, L = np.meshgrid(reserve, load)
score = (1 - np.exp(-3.2*L)) * (0.55 + 0.45*np.tanh(3.0*R))
score -= 0.45*np.maximum(L - (0.35 + 0.95*R), 0)**1.4
score -= 0.25*np.maximum(R - (0.85 - 0.35*L), 0)**1.3
score = np.clip(score, 0, 1.05)

fig, ax = plt.subplots(figsize=(10.5, 7))
im = ax.contourf(R, L, score, levels=22)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("cluster objective proxy")
ax.set_title("Capacity balance surface")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
for x0, y0, label in [(0.25,0.25,"steady"), (0.72,0.70,"scale-out"), (0.45,0.88,"shed / shorten"), (0.78,0.27,"reserve")]:
    ax.scatter([x0], [y0], s=60)
    ax.text(x0+0.03, y0+0.03, label, fontsize=13)
savefig("59_capacity_balance_surface.png")
plt.show()

## 4. Global scheduler map

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Global scheduler map", pad=18)

rounded_box(ax, (0.39, 0.67), 0.22, 0.13, "global\noptimizer", fontsize=15)
rounded_box(ax, (0.39, 0.45), 0.22, 0.13, "cluster\nstate S(t)", fontsize=15)
rounded_box(ax, (0.39, 0.23), 0.22, 0.13, "policy\nupdate", fontsize=15)

node_pos = {
    "region A\nscheduler": (0.08,0.64),
    "region B\nscheduler": (0.70,0.64),
    "region C\nscheduler": (0.08,0.28),
    "region D\nscheduler": (0.70,0.28),
}
for label, (x0,y0) in node_pos.items():
    rounded_box(ax, (x0,y0), 0.18, 0.13, label, fontsize=12)
    arrow(ax, (x0+0.18, y0+0.065), (0.39, 0.515), lw=1.4)
    arrow(ax, (0.61, 0.735), (x0+0.18 if x0 < 0.5 else x0, y0+0.105), lw=1.4, rad=0.08 if x0<0.5 else -0.08)

arrow(ax, (0.50,0.67), (0.50,0.58))
arrow(ax, (0.50,0.45), (0.50,0.36))
arrow(ax, (0.50,0.23), (0.50,0.45), rad=-0.25)
ax.text(0.5, 0.11, "Local schedulers report state; the global optimizer returns bounded policy updates.", ha="center", fontsize=13)
savefig("59_global_scheduler_map.png")
plt.show()

## 5. Node utilization balancing

In [ ]:
steps = np.arange(0, 31, 3)
target = 0.72
initial = np.array([0.92, 0.51, 0.81, 0.62, 0.38, 0.76])
rates = np.array([0.20, 0.16, 0.18, 0.14, 0.22, 0.13])
series = np.array([target + (u0-target)*np.exp(-r*steps) for u0, r in zip(initial, rates)])

fig, ax = plt.subplots(figsize=(12, 6))
for i, vals in enumerate(series, start=1):
    ax.plot(steps, vals, marker="o", label=f"node {i}")
ax.axhline(target, ls="--", color="black", label="cluster target")
ax.set_title("Node utilization balancing across the cluster")
ax.set_xlabel("coordination step")
ax.set_ylabel("GPU utilization")
ax.set_ylim(0.3, 1.0)
ax.grid(alpha=0.3)
ax.legend(ncol=3, loc="best")
savefig("59_node_utilization_balancing.png")
plt.show()

## 6. Failure recovery policy

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Failure recovery policy", pad=18)

top = ["node\nhealthy", "overload /\nfailure", "telemetry\nalert", "traffic\nreroute", "recover /\nrebalance"]
xs = np.linspace(0.06, 0.82, len(top))
w, h = 0.14, 0.14
for i, (x0, label) in enumerate(zip(xs, top)):
    rounded_box(ax, (x0, 0.62), w, h, label, fontsize=12)
    if i < len(top)-1:
        arrow(ax, (x0+w, 0.69), (xs[i+1], 0.69))

rounded_box(ax, (0.38,0.30), 0.20, 0.13, "fallback:\nlocal shorten", fontsize=12)
rounded_box(ax, (0.64,0.30), 0.20, 0.13, "global:\ncapacity shift", fontsize=12)
arrow(ax, (xs[2]+w/2,0.62), (0.48,0.43))
arrow(ax, (0.58,0.365), (0.64,0.365))
arrow(ax, (0.74,0.43), (xs[3]+w/2,0.62), rad=-0.25)
ax.text(0.5, 0.13, "Cluster optimization keeps local failures from becoming cluster-wide failures.", ha="center", fontsize=13)
savefig("59_failure_recovery_policy.png")
plt.show()

## 7. Cluster policy map

In [ ]:
reserve = np.linspace(0.05, 0.95, 120)
imbalance = np.linspace(0.05, 0.95, 120)
R, I = np.meshgrid(reserve, imbalance)

phase = np.zeros_like(R, dtype=int)
phase[(R > 0.58) & (I < 0.42)] = 1
phase[(R < 0.36) & (I > 0.55)] = 2
phase[(R >= 0.34) & (I >= 0.45)] = 3
phase[(R > 0.72) & (I > 0.75)] = 4
phase[(R < 0.22) & (I > 0.78)] = 2

labels = ["steady", "reserve", "shed/shorten", "rebalance", "scale-out"]
cmap = ListedColormap(["#440154", "#fde725", "#35b779", "#31688e", "#21918c"])
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(10.5, 7))
im = ax.imshow(phase, origin="lower", extent=[reserve.min(), reserve.max(), imbalance.min(), imbalance.max()],
               aspect="auto", cmap=cmap, norm=norm)
cbar = plt.colorbar(im, ax=ax, ticks=range(5))
cbar.ax.set_yticklabels(labels)
cbar.set_label("selected cluster policy")
ax.set_title("Cluster optimization policy map")
ax.set_xlabel("reserve capacity")
ax.set_ylabel("load imbalance")
for x0, y0, label in [(0.22,0.25,"steady"), (0.75,0.25,"reserve"), (0.22,0.72,"shed /\nshorten"), (0.58,0.62,"rebalance"), (0.82,0.86,"scale-out")]:
    ax.text(x0, y0, label, ha="center", va="center", fontsize=13)
savefig("59_cluster_policy_map.png")
plt.show()

## 8. Replica placement policy

In [ ]:
nodes = ["node A", "node B", "node C", "node D", "node E", "node F"]
parts = ["draft", "verify", "reserve", "telemetry"]
data = np.array([
    [0.50,0.22,0.18,0.10],
    [0.32,0.38,0.20,0.10],
    [0.40,0.25,0.25,0.10],
    [0.24,0.30,0.36,0.10],
    [0.55,0.16,0.19,0.10],
    [0.28,0.36,0.26,0.10],
])

fig, ax = plt.subplots(figsize=(12, 6.3))
left = np.zeros(len(nodes))
for j, part in enumerate(parts):
    bars = ax.barh(nodes, data[:, j], left=left, label=part)
    for i, b in enumerate(bars):
        if data[i, j] > 0.13:
            ax.text(left[i] + data[i,j]/2, b.get_y()+b.get_height()/2, f"{part}\n{data[i,j]:.2f}",
                    ha="center", va="center", fontsize=10)
    left += data[:, j]
ax.set_xlim(0, 1)
ax.set_title("Replica placement policy across heterogeneous nodes")
ax.set_xlabel("placement fraction")
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.25))
ax.grid(axis="x", alpha=0.25)
savefig("59_replica_placement_policy.png")
plt.show()

## 9. Repository synthesis

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Cluster optimization completes the second systems arc", pad=18)

first = [
    ("00\nContext", 0.03), ("07\nVerification\nResource", 0.16), ("13\nConfidence\nScheduling", 0.30),
    ("17\nSemi-AR\nDrafting", 0.44), ("23\nThroughput\nObjective", 0.58),
    ("29\nHardware\nConstraints", 0.72), ("37\nOperating\nRegimes", 0.86)
]
for label, x0 in first:
    rounded_box(ax, (x0, 0.66), 0.105, 0.12, label, fontsize=10)
for (_, x0), (_, x1) in zip(first[:-1], first[1:]):
    arrow(ax, (x0+0.105,0.72), (x1,0.72), lw=1.4)
ax.text(0.5,0.57,"first arc: mechanism → operating regime", ha="center", fontsize=13)
ax.plot([0.07,0.93], [0.52,0.52], color="black", lw=1.4)

second = [
    ("43\nResource\nAllocation", 0.28), ("49\nAdaptive\nInfrastructure", 0.43),
    ("53\nDistributed\nScheduling", 0.58), ("59\nCluster\nOptimization", 0.73)
]
for label, x0 in second:
    rounded_box(ax, (x0, 0.28), 0.13, 0.13, label, fontsize=10)
for (_, x0), (_, x1) in zip(second[:-1], second[1:]):
    arrow(ax, (x0+0.13,0.345), (x1,0.345), lw=1.4)
ax.text(0.5,0.15,"second arc: controller → infrastructure → distributed scheduling → cluster optimization", ha="center", fontsize=13)
savefig("59_repository_second_arc_complete.png")
plt.show()

## 10. Download figures

In [ ]:
from pathlib import Path
import zipfile

fig_dir = Path("figures")
zip_path = Path("notebook_59_cluster_optimization_figures.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(fig_dir.glob("59_*.png")):
        zf.write(path, arcname=path.name)

print(f"Created {zip_path.resolve()} with {len(list(fig_dir.glob('59_*.png')))} figures.")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Not running in Colab; use the file browser to download the zip.")

## Notebook 59 summary

Cluster optimization completes the second systems arc by turning distributed scheduling into a global serving objective. The central specification is no longer only which request receives verification, but how the whole serving cluster allocates replicas, routes load, preserves reserve, and recovers from failure without allowing local saturation to become global instability.